In [1]:
import timeit
import os
import xarray as xr
from smmregrid import cdo_generate_weights, Regridder
from cdo import Cdo
import pandas as pd
import copy
cdo = Cdo()
import dask
dask.config.set(scheduler="synchronous")

# where and which the data are
indir='../../../smmregrid/tests/data'
filelist = ['tas-healpix2.nc', 'onlytos-ipsl.nc','tas-ecearth.nc', 
            '2t-era5.nc','tos-fesom.nc', 'ua-ecearth.nc', 'mix-cesm.nc']#,'era5-mon.nc'] # the last is not available on github
#'lsm-ifs.grb'
#filelist = ['tos-fesom.nc','onlytos-ipsl.nc','tas-ecearth.nc'] 
#filelist = ['tas-ecearth.nc']
tfile = os.path.join(indir, 'r360x180.nc')

# method for remapping
methods = ['nn','con','bil']
#methods = ['con']
accesses = ['Dataset', 'DataArray']


# create an iterable dictionary, and clean cases where we know CDO does not work
defdict = {'methods': methods, 'accesses': accesses, 'extra': '', 'chunks': None}
base = {k: copy.deepcopy(defdict) for k in filelist}
if 'tos-fesom.nc' in filelist:
    base['tos-fesom.nc']['methods'].remove('bil')
if 'tas-healpix2.nc' in filelist:
    base['tas-healpix2.nc']['methods'].remove('bil')
if 'lsm-ifs.grb' in filelist:
    base['lsm-ifs.grb']['extra'] = '-setgridtype,regular'
    base['lsm-ifs.grb']['methods'].remove('bil')
    base['lsm-ifs.grb']['methods'].remove('con')
if 'mix-cesm.nc' in filelist:
    base['mix-cesm.nc']['accesses'].remove('DataArray')
if 'era5-mon.nc' in filelist:
    base['era5-mon.nc']['chunks'] = {'time': 12}
if 'ua-ecearth.nc' in filelist:
    base['ua-ecearth.nc']['chunks'] = {'plev': 3}



In [2]:
data =[]
for filein in base.keys(): 
    nr = 10

    # CDO
    wfile = cdo.gencon(tfile, input = os.path.join(indir,filein))
    ccdo = timeit.timeit(lambda: cdo.remap(tfile + ',' + wfile, input = os.path.join(indir,filein), returnXDataset = True).load(), number = nr)
    cdonoload = timeit.timeit(lambda: cdo.remap(tfile + ',' + wfile, input = os.path.join(indir,filein), returnXDataset = True), number = nr)
    #print(filein + ': Exectime CDO Remap ' + str(one/nr))

    # SMM: load field and weights, initialize regridder
    xfield = xr.open_mfdataset(os.path.join(indir,filein)).load()
    wfield = cdo_generate_weights(os.path.join(indir,filein), tfile, method = 'con').load()
    interpolator = Regridder(weights=wfield)
 
    # var as the one which have time and not have bnds, pick the first one
    myvar = [var for var in xfield.data_vars 
             if 'time' in xfield[var].dims and 'bnds' not in xfield[var].dims]
   
    # dataset infos
    nrecords = xfield[myvar[0]].shape
    nvars = len(myvar)


    sset =      timeit.timeit(lambda: interpolator.regrid(xfield).load(), number = nr)
    arr =       timeit.timeit(lambda: interpolator.regrid(xfield[myvar[0]]).load(), number = nr)
    arrnoload = timeit.timeit(lambda: interpolator.regrid(xfield[myvar[0]]), number = nr)
    #arrnomask = timeit.timeit(lambda: interpolator.regrid(xfield[myvar[0]], masked = False).load(), number = nr)
    
    setwrite =  timeit.timeit(lambda: interpolator.regrid(xfield).to_netcdf('test.nc'), number = nr)
    if os.path.isfile('test.nc'):
        os.remove('test.nc')
    arrwrite = timeit.timeit(lambda: interpolator.regrid(xfield[myvar[0]]).to_netcdf('test2.nc'), number = nr)
    if os.path.isfile('test2.nc'):
        os.remove('test2.nc')
    data.append([nvars, nrecords, ccdo, cdonoload, sset, arr, arrnoload, setwrite, arrwrite])


cnames = ['NVars', 'NRecords', 'CDO', 'CDO (NoLoad)',
          'SMM (Dataset)', 'SMM (DataArray)', 'SMM (DataArray+NoLoad)', 
          'SMM (Dataset+Write)', 'SMM (DataArray+Write)']
df = pd.DataFrame(data, index = base.keys(), columns = cnames)
final = pd.concat([df.iloc[:,0:2],df.iloc[:,2:].div(df[cnames[2]], axis=0)], join='outer', axis=1)
final



,NVars,NRecords,CDO,CDO (NoLoad),SMM (Dataset),SMM (DataArray),SMM (DataArray+NoLoad),SMM (Dataset+Write),SMM (DataArray+Write)
tas-healpix2.nc,1,"(12, 12288)",1.0,0.647083,0.074030,0.058039,0.011364,0.069243,0.065462
onlytos-ipsl.nc,1,"(12, 332, 362)",1.0,0.984145,0.132806,0.127663,0.037882,0.132958,0.130437
tas-ecearth.nc,1,"(12, 256, 512)",1.0,0.988874,0.125477,0.119095,0.030487,0.134845,0.133245
2t-era5.nc,1,"(12, 73, 144)",1.0,0.984408,0.059300,0.055256,0.015277,0.069080,0.067384
tos-fesom.nc,1,"(12, 126859)",1.0,1.000640,0.131640,0.130369,0.025743,0.141749,0.140668
ua-ecearth.nc,1,"(2, 19, 256, 512)",1.0,0.963217,0.207720,0.202131,0.061274,0.221656,0.235734
mix-cesm.nc,4,"(12, 192, 288)",1.0,0.935937,0.206668,0.072916,0.018782,0.231390,0.078316
